# 03 — Improve the model

Notebook 02 reached **74.5%** on held-out signers. Its confusion matrix showed one clear pattern:

| weak word | recall | mistaken for |
|---|---|---|
| `owie` | 38% | `bye` 23, `sick` 17, `go` 11 |
| `go` | 41% | `sleep` 17, `potty` 13, `sick` 9 |
| `finish` | 58% | `please` 28 |
| `hot` | 58% | `sad` 13, `thankyou` 12 |
| `mad` | 58% | `sick` 13, `finish` 9 |
| `sad` | 61% | `sleep` 12 |

`owie`, `sad`, `mad`, `hot` and `sick` are all made near the face. Notebook 02 threw away every `face` landmark and located the hands against the shoulder midpoint alone, so chin, forehead and chest were nearly the same place to the model. It also had no motion term: each frame stood on its own.

Three changes here:

1. **Face anchors** — wrist position relative to forehead, nose and chin, not just the shoulders.
2. **Motion features** — frame-to-frame deltas appended to each frame.
3. **Augmentation** — mirror, rotate, scale, shift, time warp, frame dropout. 6,562 training clips is thin for 20 classes.

`mom` 70% / `dad` 97% barely confused each other in notebook 02, so the body-relative idea already works. This extends it.

In [ ]:
import glob, json, os, time, warnings
import numpy as np, pandas as pd
import tensorflow as tf
warnings.filterwarnings('ignore', message='All-NaN slice encountered')

hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert hits, 'Attach the asl-signs data: Add Input -> Competitions -> asl-signs'
DATA, OUT = os.path.dirname(hits[0]), '/kaggle/working'
print('DATA =', DATA, '| GPU:', tf.config.list_physical_devices('GPU'))

## Configuration

In [ ]:
WORDS = ['hello','thankyou','please','bye','yes','no','mom','dad',
         'hungry','drink','potty','owie','sick','hot','sleep',
         'wait','go','finish','mad','sad']

T        = 32
N_HAND   = 21
POSE_LSH, POSE_RSH = 11, 12          # pose: left / right shoulder
FACE_FOREHEAD, FACE_NOSE, FACE_CHIN = 10, 1, 152   # face mesh indices
ANCHORS  = 4                          # shoulders + forehead + nose + chin
PER_HAND = N_HAND * 2 + ANCHORS * 2 + 1   # 42 coords + 8 anchor offsets + presence
BASE     = 2 * PER_HAND               # both hands
FEAT     = BASE * 2                   # base + motion deltas
TEST_IDS = [30680, 37779, 49445]
SEED     = 42

np.random.seed(SEED); tf.random.set_seed(SEED)
print(f'PER_HAND {PER_HAND}  BASE {BASE}  FEAT {FEAT}')

## Feature extraction

Per frame, per hand: the 21 landmarks translated to the wrist and divided by hand size, then the wrist offset from four anchors (shoulder midpoint, forehead, nose, chin), each divided by shoulder width, then a presence flag.

Deltas are appended after resampling, so the motion term is measured on the same 32-frame grid the model sees.

In [ ]:
USE = ['left_hand','right_hand','pose','face']

def clip_features(path):
    df = pd.read_parquet(f'{DATA}/{path}',
                         columns=['frame','type','landmark_index','x','y'])
    df = df[df['type'].isin(USE)]
    frames = np.sort(df['frame'].unique())
    if len(frames) == 0:
        return np.zeros((T, FEAT), np.float32)

    g = {k: v for k, v in df.groupby('type')}
    def grid(t, n):
        sub = g.get(t)
        a = np.full((len(frames), n, 2), np.nan, np.float32)
        if sub is None:
            return a
        fi = np.searchsorted(frames, sub['frame'].to_numpy())
        li = sub['landmark_index'].to_numpy()
        ok = li < n
        a[fi[ok], li[ok], 0] = sub['x'].to_numpy()[ok]
        a[fi[ok], li[ok], 1] = sub['y'].to_numpy()[ok]
        return a

    lh, rh = grid('left_hand', N_HAND), grid('right_hand', N_HAND)
    pose   = grid('pose', 33)
    face   = grid('face', 468)

    lsh, rsh = pose[:, POSE_LSH], pose[:, POSE_RSH]
    shoulder_mid = (lsh + rsh) / 2.0
    width = np.linalg.norm(lsh - rsh, axis=-1)
    width = np.where(np.isfinite(width) & (width > 1e-6), width, 1.0)

    anchors = [shoulder_mid,
               face[:, FACE_FOREHEAD], face[:, FACE_NOSE], face[:, FACE_CHIN]]
    # an anchor the detector lost falls back to the shoulder midpoint, then to 0
    sm = np.where(np.isfinite(shoulder_mid), shoulder_mid, 0.0)
    anchors = [np.where(np.isfinite(a), a, sm) for a in anchors]

    feats = []
    for hand in (lh, rh):
        wrist   = hand[:, 0:1, :]
        present = np.isfinite(wrist).all(-1).squeeze(-1)
        rel     = hand - wrist
        scale   = np.nanmax(np.linalg.norm(rel, axis=-1), axis=-1)
        scale   = np.where(np.isfinite(scale) & (scale > 1e-6), scale, 1.0)
        rel     = np.where(np.isfinite(rel / scale[:, None, None]),
                           rel / scale[:, None, None], 0.0)

        w = wrist.squeeze(1)
        offs = []
        for a in anchors:
            o = (w - a) / width[:, None]
            offs.append(np.where(np.isfinite(o), o, 0.0))

        feats.append(np.concatenate(
            [rel.reshape(len(frames), -1)] + offs +
            [present[:, None].astype(np.float32)], axis=1))

    x = np.concatenate(feats, axis=1).astype(np.float32)      # (F, BASE)
    idx = np.linspace(0, len(frames) - 1, T).round().astype(int)
    x = x[idx]                                                # (T, BASE)

    d = np.zeros_like(x); d[1:] = x[1:] - x[:-1]              # motion
    return np.concatenate([x, d], axis=1)                     # (T, FEAT)

In [ ]:
sub = pd.read_csv(f'{OUT}/subset.csv') if os.path.exists(f'{OUT}/subset.csv') else None
if sub is None:
    tr = pd.read_csv(f'{DATA}/train.csv')
    sub = tr[tr['sign'].isin(WORDS)].copy()
print(len(sub), 'clips')

CACHE = f'{OUT}/features_v2.npz'
if os.path.exists(CACHE):
    z = np.load(CACHE); X, y, pid = z['X'], z['y'], z['pid']
    print('loaded cache', X.shape)
else:
    t0 = time.time(); X = np.zeros((len(sub), T, FEAT), np.float32)
    for i, p in enumerate(sub['path'].to_numpy()):
        X[i] = clip_features(p)
        if i % 500 == 0:
            print(f'{i}/{len(sub)}  {time.time()-t0:.0f}s', flush=True)
    w2i = {w: i for i, w in enumerate(WORDS)}
    y   = sub['sign'].map(w2i).to_numpy().astype(np.int32)
    pid = sub['participant_id'].to_numpy()
    np.savez_compressed(CACHE, X=X, y=y, pid=pid)
    print('done', X.shape, f'{time.time()-t0:.0f}s')

In [ ]:
is_test = np.isin(pid, TEST_IDS)
Xte, yte = X[is_test], y[is_test]
Xtr, ytr, ptr = X[~is_test], y[~is_test], pid[~is_test]

tr_people = np.array(sorted(set(ptr.tolist())))
rng = np.random.default_rng(SEED); rng.shuffle(tr_people)
is_val = np.isin(ptr, tr_people[:3])
Xva, yva   = Xtr[is_val],  ytr[is_val]
Xfit, yfit = Xtr[~is_val], ytr[~is_val]
print('fit', Xfit.shape, ' val', Xva.shape, ' test', Xte.shape)

## Augmentation

Applied to the base half of the feature vector; the motion half is recomputed afterwards so the deltas stay consistent with the augmented coordinates.

**Mirror** matters most here: signers differ in handedness, and with 21 signers the model otherwise memorises which hand each person leads with.

In [ ]:
HAND = N_HAND * 2            # 42 coord values per hand
OFF  = ANCHORS * 2           # 8 anchor-offset values per hand

def augment(x):
    """x: (T, FEAT) float32. Returns an augmented copy."""
    b = x[:, :BASE].copy()
    L, R = b[:, :PER_HAND], b[:, PER_HAND:]

    # mirror: swap hands and negate every x component
    if np.random.rand() < 0.5:
        L, R = R.copy(), L.copy()
        for h in (L, R):
            h[:, 0:HAND:2]        *= -1.0      # landmark x
            h[:, HAND:HAND+OFF:2] *= -1.0      # anchor offset x

    # rotate the landmark cloud and the anchor offsets by the same small angle
    th = np.random.uniform(-0.20, 0.20)
    c, s = np.cos(th), np.sin(th)
    for h in (L, R):
        for lo, hi in ((0, HAND), (HAND, HAND + OFF)):
            xs, ys = h[:, lo:hi:2].copy(), h[:, lo+1:hi:2].copy()
            h[:, lo:hi:2]   = c * xs - s * ys
            h[:, lo+1:hi:2] = s * xs + c * ys

    # scale and shift the hand's position in the body frame
    sc = np.random.uniform(0.85, 1.15)
    dx, dy = np.random.uniform(-0.08, 0.08, 2)
    for h in (L, R):
        h[:, :HAND] *= sc
        h[:, HAND:HAND+OFF:2]   = h[:, HAND:HAND+OFF:2] * sc + dx
        h[:, HAND+1:HAND+OFF:2] = h[:, HAND+1:HAND+OFF:2] * sc + dy

    b = np.concatenate([L, R], axis=1)

    # time warp: resample the 32 frames through a random monotone curve
    if np.random.rand() < 0.5:
        p = np.random.uniform(0.75, 1.35)
        src = (np.linspace(0, 1, T) ** p * (T - 1)).round().astype(int)
        b = b[src]

    # frame dropout: blank whole frames, as a lost detection would
    drop = np.random.rand(T) < 0.08
    b[drop] = 0.0

    b += np.random.normal(0, 0.01, b.shape).astype(np.float32)

    d = np.zeros_like(b); d[1:] = b[1:] - b[:-1]
    return np.concatenate([b, d], axis=1).astype(np.float32)

# sanity check: augmentation must not change the shape or produce NaN
_a = augment(Xfit[0])
assert _a.shape == (T, FEAT) and np.isfinite(_a).all(), 'augment broken'
print('augment ok', _a.shape)

In [ ]:
BATCH = 64

def make_ds(Xa, ya, training):
    ds = tf.data.Dataset.from_tensor_slices((Xa, ya))
    if training:
        ds = ds.shuffle(len(Xa), seed=SEED, reshuffle_each_iteration=True)
        def _aug(x, lab):
            out = tf.numpy_function(augment, [x], tf.float32)
            out.set_shape((T, FEAT))
            return out, lab
        ds = ds.map(_aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(Xfit, yfit, True)
val_ds   = make_ds(Xva,  yva,  False)
print(train_ds.element_spec)

## Model

Same shape as notebook 02, a little wider, still `unroll=True` so the export stays on TFLite builtin ops.

In [ ]:
def build():
    inp = tf.keras.Input((T, FEAT))
    x = tf.keras.layers.BatchNormalization()(inp)
    x = tf.keras.layers.Bidirectional(
            tf.keras.layers.GRU(128, return_sequences=True, unroll=True))(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(128, unroll=True))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(192, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(len(WORDS), activation='softmax')(x)
    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

model = build()
print('params', model.count_params())

In [ ]:
cb = [
    tf.keras.callbacks.EarlyStopping('val_accuracy', patience=18, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=6, min_lr=1e-5),
]
hist = model.fit(train_ds, validation_data=val_ds,
                 epochs=120, callbacks=cb, verbose=2)

## Evaluation, against notebook 02

In [ ]:
loss, acc = model.evaluate(Xte, yte, verbose=0)
prob = model.predict(Xte, verbose=0); pred = prob.argmax(1)
top3 = np.mean([yte[i] in np.argsort(prob[i])[-3:] for i in range(len(yte))])

old = None
if os.path.exists(f'{OUT}/metrics.json'):
    old = json.load(open(f'{OUT}/metrics.json'))
    print(f"notebook 02 : {old['heldout_signer_accuracy']:.3f}")
print(f'notebook 03 : {acc:.3f}   (top-3 {top3:.3f}, chance {1/len(WORDS):.3f})')
if old:
    print(f"change      : {acc - old['heldout_signer_accuracy']:+.3f}")

In [ ]:
import matplotlib.pyplot as plt
cm  = tf.math.confusion_matrix(yte, pred, num_classes=len(WORDS)).numpy()
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(WORDS)), WORDS, rotation=90)
ax.set_yticks(range(len(WORDS)), WORDS)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'SignBridge v2 — held-out signers (n={len(yte)}), accuracy {acc:.1%}')
for i in range(len(WORDS)):
    for j in range(len(WORDS)):
        if cmn[i, j] > 0.02:
            ax.text(j, i, f'{cmn[i,j]*100:.0f}', ha='center', va='center',
                    fontsize=7, color='white' if cmn[i, j] > .5 else 'black')
fig.colorbar(im, shrink=.8); fig.tight_layout()
fig.savefig(f'{OUT}/confusion_matrix_v2.png', dpi=150)
plt.show()

In [ ]:
rec = dict(zip(WORDS, np.diag(cmn)))
rows = []
for w in WORDS:
    o = old['per_word_recall'][w] if old and w in old.get('per_word_recall', {}) else np.nan
    rows.append((w, round(o, 3) if o == o else None, round(rec[w], 3),
                 round(rec[w] - o, 3) if o == o else None))
cmp = pd.DataFrame(rows, columns=['word', 'v1', 'v2', 'change'])
print(cmp.sort_values('change').to_string(index=False))

print('\nremaining confusions above 8%:')
for i in range(len(WORDS)):
    for j in range(len(WORDS)):
        if i != j and cmn[i, j] > 0.08:
            print(f'  {WORDS[i]:10s} -> {WORDS[j]:10s} {cmn[i,j]:.0%}')

## Export

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tfl = conv.convert()
open(f'{OUT}/signbridge_v2.tflite', 'wb').write(tfl)
print('tflite size: %.2f MB' % (len(tfl) / 1e6))

it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
i0, o0 = it.get_input_details()[0], it.get_output_details()[0]
n_check = min(50, len(Xte)); agree = 0
for k in range(n_check):
    it.set_tensor(i0['index'], Xte[k:k+1]); it.invoke()
    agree += int(it.get_tensor(o0['index'])[0].argmax() == pred[k])
print(f'tflite agrees with keras on {agree}/{n_check} test clips')

In [ ]:
cfg = {
  'version': 2, 'words': WORDS, 'frames': T, 'features': FEAT,
  'base_features': BASE, 'per_hand': PER_HAND,
  'hand_landmarks': N_HAND,
  'pose_left_shoulder': POSE_LSH, 'pose_right_shoulder': POSE_RSH,
  'face_forehead': FACE_FOREHEAD, 'face_nose': FACE_NOSE, 'face_chin': FACE_CHIN,
  'feature_order': (
     'per frame: [LEFT: 42 hand xy normalized, 8 anchor offsets xy '
     '(shoulder_mid, forehead, nose, chin), 1 presence] then [RIGHT: same], '
     'then the same 102 values again as frame-to-frame deltas (first frame = 0)'),
  'normalization': (
     'hand: subtract wrist (landmark 0), divide by max |landmark - wrist| in that frame. '
     'anchor offset: (wrist - anchor) / shoulder_width. '
     'a missing anchor falls back to the shoulder midpoint, then to 0. '
     'missing values -> 0.0, presence flag -> 0.0'),
  'resampling': 'np.linspace(0, n_frames-1, T).round() index sampling, then deltas',
  'test_signers': TEST_IDS,
}
json.dump(cfg, open(f'{OUT}/preprocess_config_v2.json','w'), indent=2)

metrics = {'version': 2, 'heldout_signer_accuracy': float(acc), 'top3_accuracy': float(top3),
           'n_test_clips': int(len(yte)), 'n_test_signers': len(TEST_IDS),
           'n_train_clips': int(len(Xfit)), 'tflite_mb': round(len(tfl)/1e6, 2),
           'per_word_recall': {w: float(rec[w]) for w in WORDS}}
if old: metrics['v1_heldout_signer_accuracy'] = old['heldout_signer_accuracy']
json.dump(metrics, open(f'{OUT}/metrics_v2.json','w'), indent=2)
print(json.dumps({k: v for k, v in metrics.items() if k != 'per_word_recall'}, indent=2))

Download from **Output**: `signbridge_v2.tflite`, `preprocess_config_v2.json`, `confusion_matrix_v2.png`, `metrics_v2.json`.

Keep both confusion matrices. The pair — v1 beside v2, with the per-word change table — is the evidence that the diagnosis was right, and that is worth more in the mid-term than the accuracy number alone.